# Pauli SELECT

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.text_drawer import draw_text
from flagquantum.dynamic import DynamicCircuit

## `SelectPauliLCU`
A SELECT bloq for selecting and applying operators from an array of `PauliString`s.

$$
\mathrm{SELECT} = \sum_{l}|l \rangle \langle l| \otimes U_l
$$

Where $U_l$ is a member of the Pauli group.

This bloq uses the unary iteration scheme to apply `select_unitaries[selection]` to `target`
controlled on the single-bit `control` register.

#### Parameters
 - `selection_bitsize`: The size of the indexing `select` register. This should be at least `log2(len(select_unitaries))`
 - `target_bitsize`: The size of the `target` register.
 - `select_unitaries`: List of `DensePauliString`s to apply to the `target` register. Each dense pauli string must contain `target_bitsize` terms.
 - `control_val`: Optional control value. If specified, a singly controlled gate is constructed.


In [ ]:
from examples.algorithms.pauli_select_catalog import (
    PauliSelectExample, pauli_select_example_call_graph,
)

### Example Instances

In [ ]:
op = PauliSelectExample()
print(tuple(op.select.select_unitaries))

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Pauli SELECT", op.signature, {})])))

### Call Graph

In [ ]:
graph = pauli_select_example_call_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(graph.leaf_counts)
model = estimate_component_resources(op.resource_model())
print("Source model:", model)

### Compile the bounded traversal

The selection word is big-endian. The original four-row example fills its selection space. For a smaller table, bounded traversal uses the source behavior on padded addresses. Clean work follows the target register, and the optional control precedes the selection word.

In [ ]:
for cv in (None, 0, 1):
    operation = PauliSelectExample(cv)
    circuit = DynamicCircuit(operation.qubit_count)
    circuit.append(operation)
    ir = circuit.to_ir()
    counts = Counter(gate.name for gate in ir.instructions)
    print("Control:", cv, "| Emitted:", dict(counts))
    print("Source:", estimate_component_resources(operation.resource_model()))
    assert counts["t"] + counts["tdg"] == (8 if cv is None else 12)
    if cv is None:
        print(draw_text(ir, show_all_qubits=True))